Importações

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

from linearmodels.panel import PanelOLS, PooledOLS, BetweenOLS, RandomEffects
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import chi2

Carregar a base e preparar variáveis

In [2]:
# Carregar base
df = pd.read_csv("base_final.csv")

# Converter tipos
cols_num = ["ideb", "fundeb", "total", "ano"]
for col in cols_num:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Criar variável FUNDEB por aluno
df["fundeb_por_aluno"] = df["fundeb"] / df["total"]

# Tratar valores inválidos
df["fundeb_por_aluno"] = df["fundeb_por_aluno"].replace([np.inf, -np.inf], np.nan)

# Log do FUNDEB por aluno
df["ln_fundeb_por_aluno"] = np.log(df["fundeb_por_aluno"])
df["ln_fundeb_por_aluno"] = df["ln_fundeb_por_aluno"].replace([np.inf, -np.inf], np.nan)

# Base final limpa para análise
df = df.dropna(subset=["ideb", "ln_fundeb_por_aluno", "codigo_municipio", "ano"]).copy()

print("Observações após limpeza:", df.shape[0])
df.head()

Observações após limpeza: 20197


,regiao,UF,municipio,codigo_municipio,urbana,rural,total,ano,ideb,fundeb,fundeb_por_aluno,ln_fundeb_por_aluno
0,Norte,RO,alta floresta d'oeste,1100015,970,386,1356,2017,5.1,8565667.03,6316.863591,8.750978
1,Norte,RO,alto alegre dos parecis,1100379,523,561,1084,2017,5.7,7220375.26,6660.862786,8.804004
2,Norte,RO,alto paraíso,1100403,742,655,1397,2017,5.8,8294997.91,5937.722198,8.689081
3,Norte,RO,alvorada d'oeste,1100346,760,216,976,2017,5.6,6205244.17,6357.832141,8.757443
4,Norte,RO,ariquemes,1100023,5822,1208,7030,2017,5.4,42686766.39,6072.086257,8.711458


Estatísticas descritivas

In [3]:
# Selecionar variáveis descritivas
desc_vars = ["ideb", "fundeb_por_aluno", "ln_fundeb_por_aluno"]

desc = df[desc_vars].describe().T
desc["mediana"] = df[desc_vars].median()

estatisticas_descritivas = desc[["mean", "std", "min", "mediana", "max", "count"]].round(4)
estatisticas_descritivas.columns = ["Média", "Desvio-Padrão", "Mínimo", "Mediana", "Máximo", "Observações"]

print("===== ESTATÍSTICAS DESCRITIVAS =====")
print(estatisticas_descritivas)

===== ESTATÍSTICAS DESCRITIVAS =====
                          Média  Desvio-Padrão     Mínimo     Mediana  \
ideb                     5.6363         0.9653     2.3000      5.7000   
fundeb_por_aluno     12100.7418      5646.9227  2706.5696  10798.0690   
ln_fundeb_por_aluno      9.3131         0.4091     7.9034      9.2871   

                          Máximo  Observações  
ideb                     10.0000      20197.0  
fundeb_por_aluno     129484.8976      20197.0  
ln_fundeb_por_aluno      11.7713      20197.0  


In [4]:
estatisticas_descritivas.to_excel("descritiva.xlsx")
print("Arquivo exportado: descritiva.xlsx")

Arquivo exportado: descritiva.xlsx


Estrutura em painel e Estimação dos modelos

In [5]:
# Estrutura de painel
df_panel = df.set_index(["codigo_municipio", "ano"]).sort_index()

# Variáveis
y = df_panel["ideb"].astype(float)
X = df_panel[["ln_fundeb_por_aluno"]].astype(float)
X_const = sm.add_constant(X)

# Modelos
ols = sm.OLS(y, X_const).fit(cov_type="HC1")  # robusto
pooled = PooledOLS(y, X_const).fit(cov_type="robust")  # robusto
fe = PanelOLS(y, X, entity_effects=True, time_effects=True).fit(
    cov_type="clustered", cluster_entity=True
)  # cluster por município
re = RandomEffects(y, X).fit(cov_type="robust")  # robusto
between = BetweenOLS(y, X).fit(cov_type="robust")  # robusto

print("Modelos estimados com sucesso.")

Modelos estimados com sucesso.


Tabela dos modelos com coeficientes + erros-padrão robustos juntos

In [11]:
def estrelas_pvalor(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

def coef_ep(coef, ep, p):
    return f"{coef:.4f}{estrelas_pvalor(p)}\n({ep:.4f})"

tabela_modelos = pd.DataFrame({

    "OLS": [
        coef_ep(ols.params["const"], ols.bse["const"], ols.pvalues["const"]),
        coef_ep(ols.params["ln_fundeb_por_aluno"], ols.bse["ln_fundeb_por_aluno"], ols.pvalues["ln_fundeb_por_aluno"]),
        "Robusto (HC1)",
        f"{ols.rsquared:.4f}",
        f"{int(ols.nobs)}"
    ],

    "Pooled OLS": [
        coef_ep(pooled.params["const"], pooled.std_errors["const"], pooled.pvalues["const"]),
        coef_ep(pooled.params["ln_fundeb_por_aluno"], pooled.std_errors["ln_fundeb_por_aluno"], pooled.pvalues["ln_fundeb_por_aluno"]),
        "Robusto",
        f"{pooled.rsquared:.4f}",
        f"{int(pooled.nobs)}"
    ],

    "FE (two-way)": [
        "-",
        coef_ep(fe.params["ln_fundeb_por_aluno"], fe.std_errors["ln_fundeb_por_aluno"], fe.pvalues["ln_fundeb_por_aluno"]),
        "Cluster (município)",
        f"{fe.rsquared:.4f}",
        f"{int(fe.nobs)}"
    ],

    "RE": [
        "-",
        coef_ep(re.params["ln_fundeb_por_aluno"], re.std_errors["ln_fundeb_por_aluno"], re.pvalues["ln_fundeb_por_aluno"]),
        "Robusto",
        f"{re.rsquared:.4f}",
        f"{int(re.nobs)}"
    ],

    "Between": [
        "-",
        coef_ep(between.params["ln_fundeb_por_aluno"], between.std_errors["ln_fundeb_por_aluno"], between.pvalues["ln_fundeb_por_aluno"]),
        "Robusto",
        f"{between.rsquared:.4f}",
        f"{int(between.nobs)}"
    ]

}, index=[
    "Constante",
    "ln(FUNDEB por aluno)",
    "Tipo de erro-padrão",
    "R²",
    "Observações"
])

print("===== TABELA DOS MODELOS =====")
print(tabela_modelos)

===== TABELA DOS MODELOS =====
                                       OLS            Pooled OLS  \
Constante              6.5307***\n(0.1572)   6.5307***\n(0.1572)   
ln(FUNDEB por aluno)  -0.0960***\n(0.0168)  -0.0960***\n(0.0168)   
Tipo de erro-padrão          Robusto (HC1)               Robusto   
R²                                  0.0017                0.0017   
Observações                          20197                 20197   

                             FE (two-way)                   RE  \
Constante                               -                    -   
ln(FUNDEB por aluno)  0.3630***\n(0.0331)  0.5943***\n(0.0014)   
Tipo de erro-padrão   Cluster (município)              Robusto   
R²                                 0.0175               0.8968   
Observações                         20197                20197   

                                  Between  
Constante                               -  
ln(FUNDEB por aluno)  0.6025***\n(0.0014)  
Tipo de erro-padrão            

In [12]:
tabela_modelos.to_excel("modelostcc.xlsx")
print("Arquivo exportado: modelostcc.xlsx")

Arquivo exportado: modelostcc.xlsx


Testes Econométricos

In [8]:
# =========================
# 1. TESTE F DO MODELO FE
# =========================
f_stat = fe.f_statistic.stat if fe.f_statistic is not None else np.nan
f_pval = fe.f_statistic.pval if fe.f_statistic is not None else np.nan

# =========================
# 2. TESTE DE HAUSMAN
# =========================
b = fe.params
B = re.params

coef_comuns = b.index.intersection(B.index)
b = b[coef_comuns]
B = B[coef_comuns]

V_b = fe.cov.loc[coef_comuns, coef_comuns]
V_B = re.cov.loc[coef_comuns, coef_comuns]

diff = b - B
V_diff = V_b - V_B

try:
    hausman_stat = float(diff.T @ np.linalg.inv(V_diff) @ diff)
    hausman_df = len(diff)
    hausman_pval = 1 - chi2.cdf(hausman_stat, hausman_df)
except np.linalg.LinAlgError:
    hausman_stat = np.nan
    hausman_pval = np.nan

# =========================
# 3. BREUSCH-PAGAN
# =========================
bp = het_breuschpagan(ols.resid, X_const)
bp_lm_stat, bp_lm_pval, bp_f_stat, bp_f_pval = bp

print("Testes calculados com sucesso.")

Testes calculados com sucesso.


Tabela dos testes

In [9]:
tabela_testes = pd.DataFrame({
    "Estatística": [
        f_stat,
        hausman_stat,
        bp_lm_stat,
        bp_f_stat
    ],
    "p-valor": [
        f_pval,
        hausman_pval,
        bp_lm_pval,
        bp_f_pval
    ]
}, index=[
    "F (efeitos fixos)",
    "Hausman (FE vs RE)",
    "Breusch-Pagan (LM)",
    "Breusch-Pagan (F)"
]).round(4)

print("===== TABELA DOS TESTES =====")
print(tabela_testes)

===== TABELA DOS TESTES =====
                    Estatística  p-valor
F (efeitos fixos)      264.0730   0.0000
Hausman (FE vs RE)      48.7632   0.0000
Breusch-Pagan (LM)       4.2741   0.0387
Breusch-Pagan (F)        4.2746   0.0387


In [10]:
tabela_testes.to_excel("testes_TCC.xlsx")
print("Arquivo exportado: testes_TCC.xlsx")

Arquivo exportado: testes_TCC.xlsx
